# Analitika Sains Data Sepak Bola: AFC Asian Cup 2027
## Proyeksi Probabilitas 24 Negara Peserta & Peluang Timnas Indonesia Melaju ke Babak 8 Besar

Notebook ini menyajikan analisis kuantitatif komprehensif berbasis data pertandingan 4 tahun terakhir (2023–2026), evolusi nilai pasar skuad pemain diaspora, serta simulasi turnamen 100.000 iterasi Monte Carlo.

### Komposisi Grup Resmi 2027:
- **Grup A**: Arab Saudi, Kuwait, Oman, Palestina
- **Grup B**: Uzbekistan, Bahrain, Korea Utara, Yordania
- **Grup C**: Iran, Suriah, Kirgizstan, China
- **Grup D**: Australia, Tajikistan, Irak, Singapura
- **Grup E**: Korea Selatan, Uni Emirat Arab, Vietnam, Yaman
- **Grup F**: Jepang, Qatar, Thailand, Indonesia

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Konfigurasi estetika grafik bertema gelap
plt.style.use('dark_background')
DARK_BG = '#0d1117'
PANEL_BG = '#161b22'
BORDER_COLOR = '#30363d'
TEXT_COLOR = '#f0f6fc'
TEXT_MUTED = '#8b949e'
IDN_COLOR = '#e63946'
GOLD_COLOR = '#fbbf24'
CYAN_COLOR = '#38bdf8'

# Memuat berkas data
BASE_DIR = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
CSV_PATH = os.path.join(BASE_DIR, 'data', 'asian_cup_predictions.csv')
JSON_PATH = os.path.join(BASE_DIR, 'data', 'indonesia_4yr_match_analytics.json')

df_preds = pd.read_csv(CSV_PATH)
with open(JSON_PATH, 'r', encoding='utf-8') as f:
    analytics_idn = json.load(f)

print('Data prediksi 24 tim dan analitika Indonesia berhasil dimuat!')
df_preds.head(10)

### 1. Peta Peluang Lolos ke Babak 8 Besar (Perempat Final)
Membandingkan 12 negara dengan probabilitas perempat final tertinggi di Asia.

In [ ]:
top_qf = df_preds.sort_values(by='Reach_Quarter_Final_Prob(%)', ascending=False).head(12)

fig, ax = plt.subplots(figsize=(12, 6), facecolor=DARK_BG)
ax.set_facecolor(PANEL_BG)

colors = [IDN_COLOR if t == 'Indonesia' else CYAN_COLOR for t in top_qf['Team']]
bars = ax.barh(top_qf['Team'][::-1], top_qf['Reach_Quarter_Final_Prob(%)'][::-1], color=colors[::-1], edgecolor=BORDER_COLOR)

ax.set_title('PROBABILITAS LOLOS BABAK 8 BESAR (PEREMPAT FINAL) PIALA ASIA 2027', fontsize=13, weight='bold', color=TEXT_COLOR, pad=15)
ax.set_xlabel('Peluang Lolos (%)', fontsize=11, color=TEXT_COLOR)
ax.grid(color=BORDER_COLOR, linestyle='--', alpha=0.5, axis='x')

for bar in bars:
    w = bar.get_width()
    ax.text(w + 0.8, bar.get_y() + bar.get_height()/2, f'{w:.1f}%', va='center', ha='left', fontsize=10, weight='bold', color=TEXT_COLOR)

plt.tight_layout()
plt.show()

### 2. Evolusi Nilai Pasar Skuad Timnas Indonesia (2023–2026)
Lonjakan nilai pasar dari €5,85M menjadi €36,5M yang mendongkrak daya saing Garuda ke peringkat 6 Asia.

In [ ]:
yearly = analytics_idn['squad_market_value_evolution']['yearly_trajectory']
years = [d['year'].split(' (')[0] for d in yearly]
vals = [d['market_value_eur'] / 1e6 for d in yearly]
ranks = [d['rank_in_asia'] for d in yearly]

fig, ax = plt.subplots(figsize=(10, 5), facecolor=DARK_BG)
ax.set_facecolor(PANEL_BG)

ax.plot(years, vals, marker='o', color=IDN_COLOR, linewidth=3, markersize=8, label='Nilai Pasar Skuad (€ Juta)')
ax.fill_between(years, vals, color=IDN_COLOR, alpha=0.2)

for x, y, r in zip(years, vals, ranks):
    ax.text(x, y + 1.2, f'€{y:.1f}M\n(Peringkat #{r} Asia)', ha='center', va='bottom', fontsize=9.5, weight='bold', color=TEXT_COLOR)

ax.set_title('TRANSFORMASI NILAI PASAR SKUAD TIMNAS INDONESIA (2023–2026)', fontsize=12, weight='bold', color=TEXT_COLOR, pad=15)
ax.set_ylabel('Nilai Pasar Skuad (€ Juta)', fontsize=10, color=TEXT_COLOR)
ax.set_ylim(0, 45)
ax.grid(color=BORDER_COLOR, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

### 3. Bedah 3 Skenario Timnas Indonesia Menuju Babak 8 Besar dari Grup F
Grup F: Jepang, Qatar, Thailand, Indonesia.

### 4. Analisis Mikro X-Factor & Pengali Keberuntungan (24 Negara Peserta)
Evaluasi pemain paling berpengaruh (*Clutch Score* & *Luck Multiplier*) penentu laga genting fase gugur.

In [ ]:
MICRO_PATH = os.path.join(BASE_DIR, 'data', 'all_24_teams_squad_micro_analytics.json')
with open(MICRO_PATH, 'r', encoding='utf-8') as f:
    raw_micro = json.load(f)

teams_dict = raw_micro.get('teams', raw_micro)
clutch_list = []
for team_name, tinfo in teams_dict.items():
    kp = tinfo.get('key_players_micro_xfactor', [])
    top_clutch = max([p.get('clutch_score_100', 0) for p in kp]) if kp else 0
    top_luck = max([p.get('micro_luck_multiplier', 1.0) for p in kp]) if kp else 1.0
    p_names = ', '.join([p.get('name', '') for p in kp])
    clutch_list.append({
        'Team': team_name,
        'Group': tinfo.get('group', ''),
        'Top_Clutch_Score': top_clutch,
        'Max_Luck_Multiplier': top_luck,
        'Key_Players': p_names,
        'Matches_Post_2023': tinfo.get('post_asian_cup_record_to_oct_2026', {}).get('total_matches', 0),
        'Win_Rate_Post_2023': tinfo.get('post_asian_cup_record_to_oct_2026', {}).get('win_rate_pct', 0.0)
    })

df_clutch = pd.DataFrame(clutch_list).sort_values(by='Top_Clutch_Score', ascending=False)
print(f'Data analitika mikro untuk {len(df_clutch)} negara berhasil diolah.')
df_clutch.head(10)

### 5. Komparasi Bar Chart: Clutch Score Tertinggi 24 Negara
Menunjukkan seberapa tajam faktor pembeda individu saat laga memasuki menit ke-80+ atau adu penalti.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 7), facecolor=DARK_BG)
ax.set_facecolor(PANEL_BG)

bar_colors = [IDN_COLOR if t == 'Indonesia' else ('#60a5fa' if s >= 90 else '#94a3b8') for t, s in zip(df_clutch['Team'], df_clutch['Top_Clutch_Score'])]
bars = ax.bar(df_clutch['Team'], df_clutch['Top_Clutch_Score'], color=bar_colors, edgecolor=BORDER_COLOR)

ax.set_title('DISTRIBUSI SKOR CLUTCH PEMAIN KUNCI 24 NEGARA PIALA ASIA 2027', fontsize=13, weight='bold', color=TEXT_COLOR, pad=15)
ax.set_ylabel('Clutch Score (0 - 100)', fontsize=11, color=TEXT_COLOR)
plt.xticks(rotation=45, ha='right', color=TEXT_COLOR, fontsize=9.5)
ax.grid(color=BORDER_COLOR, linestyle='--', alpha=0.5, axis='y')
ax.set_ylim(50, 105)

for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 1, f'{int(h)}', ha='center', va='bottom', fontsize=8.5, weight='bold', color=TEXT_COLOR)

plt.tight_layout()
plt.show()

### 6. Rekor Lengkap Timnas Indonesia Pasca Piala Asia 2023 s.d. Oktober 2026 (27 Laga Resmi)
Tanpa batas 20 laga sebelumnya: menyajikan tren performa menyeluruh hingga jeda internasional Oktober 2026.

In [ ]:
post_matches = analytics_idn.get('matches_post_asian_cup_history', [])
df_post = pd.DataFrame(post_matches)
summary_stats = analytics_idn.get('matches_post_asian_cup_summary', {})
print('Ringkasan Statistik 27 Laga Resmi Pasca Piala Asia 2023:')
for k, v in summary_stats.items():
    print(f'  - {k}: {v}')

cols_to_show = [c for c in ['date', 'competition', 'opponent', 'score', 'result', 'clean_sheet', 'notes'] if c in df_post.columns]
df_post[cols_to_show].tail(10)